In [1]:
"""
NETWORK OPTIMIZATION (SUPPLY CHAIN) — Pyomo MILP Template
--------------------------------------------------------
What this model does (business story):

We have:
  - Plants (make products)
  - DCs (store and ship products)
  - Customers (need products)
  - Multiple products
  - Multiple time periods

Decisions:
  1) Open which plants and DCs (binary: 0/1)
  2) Produce how much at each plant (by product, by time)
  3) Ship how much plant->DC and DC->customer (by product, by time)
  4) Hold inventory at DCs (by product, by time)
  5) Optionally allow unmet demand with a BIG penalty (backorder/lost sales proxy)

Objective:
  Minimize total cost = fixed opening + production + transport + holding + unmet demand penalty

Constraints:
  - Plant capacity (only if open)
  - DC throughput capacity (only if open)
  - Flow conservation at DC (inventory balance)
  - Demand satisfaction (shipments + unmet >= demand)
  - Optional CO2 budget (cap)
"""

from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals, Binary,
    Constraint, Objective, minimize, value, SolverFactory
)

def build_network_design_model(data: dict) -> ConcreteModel:
    m = ConcreteModel(name="SC_Network_Design_MultiPeriod")

    # ----------------------------
    # 1) SETS
    # ----------------------------
    m.P = Set(initialize=data["PLANTS"])      # plants
    m.D = Set(initialize=data["DCS"])         # distribution centers
    m.C = Set(initialize=data["CUSTOMERS"])   # customers
    m.K = Set(initialize=data["PRODUCTS"])    # products
    m.T = Set(initialize=data["PERIODS"], ordered=True)  # periods (ordered helps with inventory carryover)

    # For convenience: previous period mapping
    periods = list(data["PERIODS"])
    prev_map = {}
    for i, t in enumerate(periods):
        prev_map[t] = periods[i-1] if i > 0 else None

    # ----------------------------
    # 2) PARAMETERS (inputs)
    # ----------------------------
    # Fixed opening costs
    m.f_open_plant = Param(m.P, initialize=data["fixed_open_plant"], within=NonNegativeReals)
    m.f_open_dc    = Param(m.D, initialize=data["fixed_open_dc"], within=NonNegativeReals)

    # Capacities (if open)
    # Plant capacity is usually "production-hours" or "units" per period.
    m.cap_plant = Param(m.P, m.T, initialize=data["cap_plant"], within=NonNegativeReals)
    # DC throughput capacity: how many units can pass through per period (inbound or outbound).
    m.cap_dc    = Param(m.D, m.T, initialize=data["cap_dc"], within=NonNegativeReals)

    # Unit production cost
    m.c_prod = Param(m.P, m.K, initialize=data["prod_cost"], within=NonNegativeReals)

    # Transport costs
    m.c_pd = Param(m.P, m.D, initialize=data["ship_cost_pd"], within=NonNegativeReals)  # plant -> dc
    m.c_dc = Param(m.D, m.C, initialize=data["ship_cost_dc"], within=NonNegativeReals)  # dc -> customer

    # Inventory holding cost at DC
    m.c_hold = Param(m.D, m.K, initialize=data["hold_cost"], within=NonNegativeReals)

    # Demand
    m.dem = Param(m.C, m.K, m.T, initialize=data["demand"], within=NonNegativeReals)

    # Big penalty for unmet demand (so the model *hates* unmet demand)
    m.c_unmet = Param(initialize=data.get("unmet_penalty", 1_000.0), within=NonNegativeReals)

    # Optional CO2 parameters (set co2_budget=None to disable)
    co2_budget = data.get("co2_budget", None)
    if co2_budget is not None:
        m.co2_budget = Param(initialize=float(co2_budget), within=NonNegativeReals)
        # kg CO2 per shipped unit on each lane
        m.e_pd = Param(m.P, m.D, initialize=data["co2_pd"], within=NonNegativeReals)
        m.e_dc = Param(m.D, m.C, initialize=data["co2_dc"], within=NonNegativeReals)

    # Initial inventory at DCs
    m.inv0 = Param(m.D, m.K, initialize=data.get("init_inventory", {}), default=0.0, within=NonNegativeReals)

    # ----------------------------
    # 3) DECISION VARIABLES
    # ----------------------------
    # Open decisions (design)
    m.yP = Var(m.P, domain=Binary)  # open plant?
    m.yD = Var(m.D, domain=Binary)  # open DC?

    # Production and shipment (flow)
    m.prod = Var(m.P, m.K, m.T, domain=NonNegativeReals)     # production units
    m.xPD  = Var(m.P, m.D, m.K, m.T, domain=NonNegativeReals) # ship plant->dc
    m.xDC  = Var(m.D, m.C, m.K, m.T, domain=NonNegativeReals) # ship dc->customer

    # Inventory at DC
    m.inv = Var(m.D, m.K, m.T, domain=NonNegativeReals)

    # Optional unmet demand (keeps model feasible, but punished)
    m.unmet = Var(m.C, m.K, m.T, domain=NonNegativeReals)

    # ----------------------------
    # 4) CONSTRAINTS
    # ----------------------------

    # 4.1 Plant capacity: total production across products <= cap * open_flag
    def plant_cap_rule(m, p, t):
        return sum(m.prod[p, k, t] for k in m.K) <= m.cap_plant[p, t] * m.yP[p]
    m.PlantCapacity = Constraint(m.P, m.T, rule=plant_cap_rule)

    # 4.2 Production must leave the plant (ship to DCs)
    #     (If you want allow plant inventory, you'd add plant inventory variables. Keep simple here.)
    def plant_flow_rule(m, p, k, t):
        return sum(m.xPD[p, d, k, t] for d in m.D) == m.prod[p, k, t]
    m.PlantFlow = Constraint(m.P, m.K, m.T, rule=plant_flow_rule)

    # 4.3 DC throughput capacity:
    #     Outbound shipments across all customers & products <= cap_dc * open_flag
    def dc_cap_rule(m, d, t):
        return sum(m.xDC[d, c, k, t] for c in m.C for k in m.K) <= m.cap_dc[d, t] * m.yD[d]
    m.DcCapacity = Constraint(m.D, m.T, rule=dc_cap_rule)

    # 4.4 DC inventory balance (flow conservation):
    #     inventory_end = inventory_start + inbound - outbound
    def dc_balance_rule(m, d, k, t):
        prev_t = prev_map[t]
        inv_start = m.inv0[d, k] if prev_t is None else m.inv[d, k, prev_t]

        inbound  = sum(m.xPD[p, d, k, t] for p in m.P)
        outbound = sum(m.xDC[d, c, k, t] for c in m.C)

        return m.inv[d, k, t] == inv_start + inbound - outbound
    m.DcBalance = Constraint(m.D, m.K, m.T, rule=dc_balance_rule)

    # 4.5 Demand satisfaction:
    #     total shipped to customer + unmet == demand
    def demand_rule(m, c, k, t):
        shipped = sum(m.xDC[d, c, k, t] for d in m.D)
        return shipped + m.unmet[c, k, t] == m.dem[c, k, t]
    m.Demand = Constraint(m.C, m.K, m.T, rule=demand_rule)

    # 4.6 Optional: if a DC is closed, it can't ship out (tighter than capacity)
    #     This helps numerics and makes logic explicit:
    def dc_ship_open_rule(m, d, c, k, t):
        # if yD[d] = 0 -> xDC <= 0; if yD[d] = 1 -> xDC <= bigM
        # Use a safe bigM based on total demand in that period for that product.
        bigM = sum(value(m.dem[c2, k, t]) for c2 in m.C)
        return m.xDC[d, c, k, t] <= bigM * m.yD[d]
    m.DcShipIfOpen = Constraint(m.D, m.C, m.K, m.T, rule=dc_ship_open_rule)

    # 4.7 Optional: CO2 budget (cap)
    if co2_budget is not None:
        def co2_rule(m):
            ship_pd = sum(m.e_pd[p, d] * m.xPD[p, d, k, t] for p in m.P for d in m.D for k in m.K for t in m.T)
            ship_dc = sum(m.e_dc[d, c] * m.xDC[d, c, k, t] for d in m.D for c in m.C for k in m.K for t in m.T)
            return ship_pd + ship_dc <= m.co2_budget
        m.CO2Budget = Constraint(rule=co2_rule)

    # ----------------------------
    # 5) OBJECTIVE (minimize total cost)
    # ----------------------------
    def total_cost(m):
        fixed = sum(m.f_open_plant[p] * m.yP[p] for p in m.P) + sum(m.f_open_dc[d] * m.yD[d] for d in m.D)

        production = sum(m.c_prod[p, k] * m.prod[p, k, t] for p in m.P for k in m.K for t in m.T)

        transport = (
            sum(m.c_pd[p, d] * m.xPD[p, d, k, t] for p in m.P for d in m.D for k in m.K for t in m.T)
            + sum(m.c_dc[d, c] * m.xDC[d, c, k, t] for d in m.D for c in m.C for k in m.K for t in m.T)
        )

        holding = sum(m.c_hold[d, k] * m.inv[d, k, t] for d in m.D for k in m.K for t in m.T)

        unmet_pen = sum(m.c_unmet * m.unmet[c, k, t] for c in m.C for k in m.K for t in m.T)

        return fixed + production + transport + holding + unmet_pen

    m.Obj = Objective(rule=total_cost, sense=minimize)
    return m


if __name__ == "__main__":
    # -------------------------------------------------------------------------
    # EXAMPLE DATA (small demo) — replace with your real tables later
    # -------------------------------------------------------------------------
    PLANTS    = ["PL1", "PL2"]
    DCS       = ["DC1", "DC2", "DC3"]
    CUSTOMERS = ["C1", "C2", "C3", "C4"]
    PRODUCTS  = ["A", "B"]
    PERIODS   = [1, 2, 3]   # 3 months

    # Helper to create (p,t) or (d,t) capacity dicts
    cap_plant = {(p, t): (800 if p == "PL1" else 600) for p in PLANTS for t in PERIODS}
    cap_dc    = {(d, t): (900 if d in ["DC1", "DC2"] else 500) for d in DCS for t in PERIODS}

    fixed_open_plant = {"PL1": 20_000, "PL2": 15_000}
    fixed_open_dc    = {"DC1": 10_000, "DC2": 9_000, "DC3": 6_000}

    prod_cost = {("PL1", "A"): 4.0, ("PL1", "B"): 5.0,
                 ("PL2", "A"): 4.2, ("PL2", "B"): 4.8}

    # Costs by lane (keep simple but complete)
    ship_cost_pd = {(p, d): 1.0 + 0.3 * (i + j)
                    for i, p in enumerate(PLANTS)
                    for j, d in enumerate(DCS)}

    ship_cost_dc = {(d, c): 1.2 + 0.2 * (i + j)
                    for i, d in enumerate(DCS)
                    for j, c in enumerate(CUSTOMERS)}

    hold_cost = {(d, k): (0.25 if d != "DC3" else 0.18) for d in DCS for k in PRODUCTS}

    # Demand (customer, product, period)
    demand = {}
    for c in CUSTOMERS:
        for k in PRODUCTS:
            for t in PERIODS:
                base = 120 if k == "A" else 80
                seasonal = 20 * (t - 1)  # rising demand
                demand[(c, k, t)] = base + seasonal

    # Optional CO2 (kg per unit shipped)
    co2_pd = {(p, d): 0.4 + 0.1 * j for p in PLANTS for j, d in enumerate(DCS)}
    co2_dc = {(d, c): 0.3 + 0.05 * j for d in DCS for j, c in enumerate(CUSTOMERS)}

    data = {
        "PLANTS": PLANTS,
        "DCS": DCS,
        "CUSTOMERS": CUSTOMERS,
        "PRODUCTS": PRODUCTS,
        "PERIODS": PERIODS,

        "fixed_open_plant": fixed_open_plant,
        "fixed_open_dc": fixed_open_dc,

        "cap_plant": cap_plant,
        "cap_dc": cap_dc,

        "prod_cost": prod_cost,
        "ship_cost_pd": ship_cost_pd,
        "ship_cost_dc": ship_cost_dc,
        "hold_cost": hold_cost,
        "demand": demand,

        # If you want to force full demand satisfaction, set this HUGE (or remove unmet var + use >=).
        "unmet_penalty": 10_000.0,

        # Turn CO2 cap ON by setting a number, OFF by setting None
        "co2_budget": 2500.0,
        "co2_pd": co2_pd,
        "co2_dc": co2_dc,

        # initial inventory (optional)
        "init_inventory": {("DC1", "A"): 50, ("DC1", "B"): 30}
    }

    m = build_network_design_model(data)

    # -------------------------------------------------------------------------
    # SOLVE
    # -------------------------------------------------------------------------
    # Good free options: "highs" (recommended), "cbc", "glpk"
    # Make sure you have one installed in your environment.
    solver_name = "highs"  # change to "cbc" or "glpk" if needed
    solver = SolverFactory(solver_name)
    res = solver.solve(m, tee=False)

    print("\n=== STATUS ===")
    print(res.solver.status, res.solver.termination_condition)

    print("\n=== OPEN FACILITIES ===")
    for p in m.P:
        print(f"Open Plant {p}: {value(m.yP[p])}")
    for d in m.D:
        print(f"Open DC    {d}: {value(m.yD[d])}")

    print("\n=== COST ===")
    print("Total cost:", value(m.Obj))

    # Example: show total unmet demand (should be 0 if penalty is high enough and capacities allow)
    unmet_total = sum(value(m.unmet[c, k, t]) for c in m.C for k in m.K for t in m.T)
    print("\nTotal unmet demand:", unmet_total)



=== STATUS ===
ok optimal

=== OPEN FACILITIES ===
Open Plant PL1: 1.0
Open Plant PL2: 1.0
Open DC    DC1: 1.0
Open DC    DC2: 0.0
Open DC    DC3: 1.0

=== COST ===
Total cost: 70818.0

Total unmet demand: 0.0
